# Expanded Khmer OCR selected demo
Choose a model and an image, then click **Run real prediction**. These are deliberately selected exact-match examples. This does not measure overall accuracy or guarantee performance on new images. CPU is sufficient.

In [ ]:
%pip -q install torch pillow numpy huggingface_hub ipywidgets
from huggingface_hub import hf_hub_download
from pathlib import Path
import sys, zipfile, json
repo='Seoul12/khmer-ocr-three-architectures'
module=hf_hub_download(repo,'inference_three.py')
sys.path.insert(0,str(Path(module).parent))
from inference_three import MultiRecognizer
archive=hf_hub_download('Seoul12/khmer-ocr-selected-demo','Expanded_Khmer_OCR_Demo.zip',repo_type='dataset')
with zipfile.ZipFile(archive) as z:z.extractall('expanded_demo')
root=Path('expanded_demo'); data=json.loads((root/'manifest.json').read_text())
print(data['summary'])


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
models={}
model=widgets.Dropdown(options=[('Transformer','transformer'),('BiLSTM','bilstm'),('BiGRU','bigru')],value='transformer',description='Model:')
image=widgets.Dropdown(description='Image:',layout=widgets.Layout(width='90%'))
run=widgets.Button(description='Run real prediction',button_style='primary')
output=widgets.Output()
def update_images(change=None):
    candidates=[r for r in data['images'] if model.value in r['exact_models']]
    image.options=[('Row '+str(r['source_row_index'])+' — '+r['answer'][:40],r['image']) for r in candidates]
    with output:
        clear_output(wait=True)
        print(len(candidates),'selected exact-match images for this model. Click Run real prediction.')
def predict(_):
    with output:
        clear_output(wait=True)
        if image.value is None: print('No selected images for this model.'); return
        name=model.value
        if name not in models: models[name]=MultiRecognizer(hf_hub_download(repo,name+'.pt'))
        row=next(r for r in data['images'] if r['image']==image.value)
        path=root/row['image']; display(Image.open(path))
        prediction=models[name].predict(path)
        print('ANSWER:',row['answer']);print('PREDICTION:',prediction)
        print('RIGHT — exact match' if prediction==row['answer'] else 'WRONG — review runtime/checkpoint/preprocessing')
        print('Selected demonstration example; not overall accuracy.')
model.observe(update_images,names='value');run.on_click(predict)
update_images();display(model,image,run,output)
